In [ ]:
! pip install keras-tuner

In [ ]:
! pip install tensorflow

In [ ]:
! pip install scikit-learn

In [ ]:
! pip install pandas

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
data=pd.read_csv('huge_1M_titanic.csv')

In [ ]:
data=data.sample(10000,random_state=42)

In [ ]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
200330,201640,0,3,"Name201640, Mr. Surname201640",male,53.0,0,0,2700,13.250337,NaN,Q
85740,87050,0,3,"Name87050, Mr. Surname87050",male,32.0,0,0,345769,5.494364,NaN,S
42490,43800,0,2,"Name43800, Mr. Surname43800",male,20.0,0,0,PC 17760,8.879396,D19,S
119514,120824,0,1,"Name120824, Mr. Surname120824",male,55.0,0,0,248733,73.525558,C30,S
129273,130583,0,3,"Name130583, Mrs. Surname130583",female,NaN,1,4,7545,25.835379,NaN,S


In [ ]:
data=data.drop(['PassengerId','Name','Age','Ticket','Cabin'],axis=1)

In [ ]:
data['Embarked']=data['Embarked'].replace({'S':'Southampton','C':'Cherbourg','Q':'Queenstown'})

In [ ]:
data.dropna(subset=['Embarked'],inplace=True)

In [ ]:
data=data.reset_index(drop=True)

In [ ]:
data['Fare']=data['Fare'].astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label=LabelEncoder()

In [ ]:
data['Sex']=label.fit_transform(data['Sex'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
onehot=OneHotEncoder(sparse_output=False)

In [ ]:
Embarked=onehot.fit_transform(data[['Embarked']])

In [ ]:
Embarked=pd.DataFrame(Embarked,columns=onehot.get_feature_names_out())

In [ ]:
Embarked=Embarked.reset_index(drop=True)

In [ ]:
data=pd.concat([data.drop(columns=['Embarked']),Embarked],axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scale=StandardScaler()

In [ ]:
num_cols=['Pclass', 'SibSp', 'Parch', 'Fare']

In [ ]:
data[num_cols]=scale.fit_transform(data[num_cols])

In [ ]:
data.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
Embarked_Cherbourg,0
Embarked_Queenstown,0
Embarked_Southampton,0


In [ ]:
X=data.drop(columns=['Survived'])

In [ ]:
Y=data['Survived']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
X_train,X_valid,Y_train,Y_valid=train_test_split(X_train,Y_train,test_size=0.2,random_state=42)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model=Sequential([Dense(128,input_shape=(X_train.shape[1],),activation='relu'),Dense(64,activation='relu'),Dense(32,activation='relu'),Dense(1,activation='sigmoid')]) #First Hidden Layer with 128 neurons, input shape is number of features in training data, activation function is relu. Second Hidden Layer with 64 neurons, activation function is relu. Third Hidden Layer with 32 neurons, activation function is relu. Output Layer with 1 neuron and sigmoid activation for classification. First layer has more neuron to capture more basic features and subsequent layers have fewer neurons to reduce overfitting. Output layer has 1 neuron as we have binary classification problem. We use comma with input shape because it is a tuple syntax to represent dimension like 4, represent one dimension(column) with 4 values. We use relu activation function for hidden layers as it is most commonly used and sigmoid for output layer as we have binary classification problem.

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
import tensorflow as tf

In [ ]:
import tensorflow as tf
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
tf.keras.losses.BinaryFocalCrossentropy


keras.src.losses.losses.BinaryFocalCrossentropy

In [ ]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy']) #if we use optimizer as adam directly, it will take default learning rate of 0.001 which is very low and will take more time to converge. So we have defined our own learning rate of 0.01 and binary_crossentropy is used as loss function for binary classification problem. Accuracy is used as metric to evaluate the model performance.

In [ ]:
model.fit(X_train,Y_train,validation_data=(X_valid,Y_valid),epochs=50)

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.8075 - loss: 0.4332 - val_accuracy: 0.8245 - val_loss: 0.3997
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8255 - loss: 0.4001 - val_accuracy: 0.8345 - val_loss: 0.3937
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8316 - loss: 0.3926 - val_accuracy: 0.8502 - val_loss: 0.3755
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8379 - loss: 0.3791 - val_accuracy: 0.8382 - val_loss: 0.3807
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8399 - loss: 0.3716 - val_accuracy: 0.8426 - val_loss: 0.3800
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8435 - loss: 0.3691 - val_accuracy: 0.8527 - val_loss: 0.3502
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8431 - loss: 0.3654 - val_accuracy: 0.8464 - val_loss: 0.3640
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8454 - loss: 0.3600 - val_accuracy: 

# Hyperparameter Tuning.
1)Optimal optimizers

In [ ]:
def build_model(hp):
  model=Sequential([Dense(128,input_shape=(X_train.shape[1],),activation='relu'),Dense(64,activation='relu'),Dense(32,activation='relu'),Dense(1,activation='sigmoid')]) #First Hidden Layer with 128 neurons, input shape is number of features in training data, activation function is relu. Second Hidden Layer with 64 neurons, activation function is relu. Third Hidden Layer with 32 neurons, activation function is relu. Output Layer with 1 neuron and sigmoid activation for classification. First layer has more neuron to capture more basic features and subsequent layers have fewer neurons to reduce overfitting. Output layer has 1 neuron as we have binary classification problem. We use comma with input shape because it is a tuple syntax to represent dimension like 4, represent one dimension(column) with 4 values. We use relu activation function for hidden layers as it is most commonly used and sigmoid for output layer as we have binary classification problem.
  optimizer=hp.Choice('optimizer',values=['Adam','sgd','rmsprop','AdamW','AdamDelta','AdaGrad','Adamax'])
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [ ]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5) #we want high validation accuracy and we are using keras tuner.

Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
tuner.search(X_train,Y_train,validation_data=(X_test,Y_test),epochs=10)

In [ ]:
tuner.get_best_hyperparameters()[0].values
#

{'optimizer': 'AdamW'}

In [ ]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,initial_epoch=11)


Epoch 12/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.8457 - loss: 0.3572 - val_accuracy: 0.8425 - val_loss: 0.3488
Epoch 13/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8464 - loss: 0.3533 - val_accuracy: 0.8410 - val_loss: 0.3442
Epoch 14/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8457 - loss: 0.3528 - val_accuracy: 0.8501 - val_loss: 0.3408
Epoch 15/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8493 - loss: 0.3497 - val_accuracy: 0.8480 - val_loss: 0.3430
Epoch 16/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8481 - loss: 0.3464 - val_accuracy: 0.8506 - val_loss: 0.3417
Epoch 17/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8508 - loss: 0.3445 - val_accuracy: 0.8480 - val_loss: 0.3407
Epoch 18/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8539 - loss: 0.3409 - val_accuracy: 0.8460 - val_loss: 0.3337
Epoch 19/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8515 - loss: 0.3393 - 

2)Choosing number of nodes in Hidden Layer and activation function

In [ ]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)
from tensorflow.keras.layers import Input

In [ ]:
def build_model(hp):

    nodes=hp.Int('nodes',8,128,step=8)
    activation_func=hp.Choice('activation',values=['sigmoid','tanh'])
    model=Sequential()
    model.add(Input(shape=((X_train.shape[1],))))
    model.add(Dense(nodes,activation='relu'))
    model.add(Dense(nodes,activation='relu'))
    model.add(Dense(1,activation=activation_func))

    model.compile(optimizer='rmsprop',metrics=['accuracy'],loss='binary_crossentropy')

    return model

In [ ]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='nodes_2',project_name='nodes_details')

Reloading Tuner from nodes_2/nodes_details/tuner0.json


In [ ]:
tuner.search(X_train,Y_train,validation_data=(X_test,Y_test),epochs=10)

Trial 5 Complete [00h 01m 26s]
val_accuracy: 0.8411823511123657

Best val_accuracy So Far: 0.8411823511123657
Total elapsed time: 00h 07m 36s


In [ ]:
tuner.get_best_hyperparameters()[0].values

{'nodes': 48, 'activation': 'tanh'}

In [ ]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [64]:
model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,initial_epoch=11)


Epoch 12/100
  1/200 ━━━━━━━━━━━━━━━━━━━━ 24s 122ms/step - accuracy: 0.8438 - loss: 0.5086

/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8388 - loss: 0.4903

/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.8381 - loss: 0.4530 - val_accuracy: 0.8372 - val_loss: 0.3877
Epoch 13/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8424 - loss: 0.4202 - val_accuracy: 0.8372 - val_loss: 0.3971
Epoch 14/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.8403 - loss: 0.4086 - val_accuracy: 0.8432 - val_loss: 0.3678
Epoch 15/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.8410 - loss: 0.4105 - val_accuracy: 0.8357 - val_loss: 0.4426
Epoch 16/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8365 - loss: 0.4393 - val_accuracy: 0.8382 - val_loss: 0.3964
Epoch 17/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8370 - loss: 0.4357 - val_accuracy: 0.8292 - val_loss: 0.4037
Epoch 18/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.8412 - loss: 0.4256 - val_accuracy: 0.8402 - val_loss: 0.3985
Epoch 19/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8390 - loss: 0.4584 - val_a

3) No. of Optimal hidden Layers

In [65]:
def build_model(hp):

    model=Sequential()
    model.add(Input(shape=(X_train.shape[1],)))

    for i in range(hp.Int('hidden',min_value=1,max_value=10,step=1)):
        model.add(Dense(88,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [66]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='hidden')

In [67]:
tuner.search(X_train,Y_train,validation_data=(X_test,Y_test),epochs=10)

Trial 5 Complete [00h 02m 18s]
val_accuracy: 0.8436873555183411

Best val_accuracy So Far: 0.8567134141921997
Total elapsed time: 00h 15m 39s


In [68]:
tuner.get_best_hyperparameters()[0].values

{'hidden': 4}

In [69]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [70]:
model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,initial_epoch=11)


Epoch 12/100


/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.8459 - loss: 0.3389

/usr/local/lib/python3.12/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 76ms/step - accuracy: 0.8525 - loss: 0.3396 - val_accuracy: 0.8452 - val_loss: 0.3339
Epoch 13/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 77ms/step - accuracy: 0.8507 - loss: 0.3364 - val_accuracy: 0.8432 - val_loss: 0.3398
Epoch 14/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 76ms/step - accuracy: 0.8576 - loss: 0.3319 - val_accuracy: 0.8532 - val_loss: 0.3434
Epoch 15/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 22s 86ms/step - accuracy: 0.8575 - loss: 0.3337 - val_accuracy: 0.8522 - val_loss: 0.3320
Epoch 16/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - accuracy: 0.8562 - loss: 0.3305 - val_accuracy: 0.8587 - val_loss: 0.3238
Epoch 17/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 74ms/step - accuracy: 0.8579 - loss: 0.3266 - val_accuracy: 0.8527 - val_loss: 0.3239
Epoch 18/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 74ms/step - accuracy: 0.8576 - loss: 0.3273 - val_accuracy: 0.8617 - val_loss: 0.3240
Epoch 19/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - accuracy: 0.8590 - loss: 0.3281

Hyerparameter Tuning-all in one, also including dropout layers

In [71]:
def build_model(hp):
    optimizer_hp=hp.Choice('optimizer',values=['SGD','RMSprop','Adam','AdamW','Adadelta','Adagrad','Adamax'])

    model=Sequential()
    model.add(Input(shape=(X_train.shape[1],)))

    for i in range(hp.Int('hidden',min_value=1,max_value=10,step=1)):
        nodes=hp.Int('nodes',min_value=8,max_value=128,step=8)
        model.add(Dense(nodes,activation='relu'))

        dropout_val=hp.Float('dropout',min_value=0.1,max_value=0.9,step=0.1)
        model.add(Dropout(dropout_val))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=optimizer_hp,loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [72]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='all_in_one')

In [73]:
tuner.search(X_train,Y_train,validation_data=(X_test,Y_test),epochs=10)

Trial 5 Complete [00h 03m 11s]
val_accuracy: 0.8166332840919495

Best val_accuracy So Far: 0.8226453065872192
Total elapsed time: 00h 14m 08s


In [79]:
!ls

 all_in_one   huge_1M_titanic.csv	    nodes_2	  untitled_project
 hidden      'Hyperparameter_Tuning.zip '   sample_data


In [80]:
!zip -r "ANN_Project.zip" all_in_one huge_1M_titanic.csv nodes_2 untitled_project hidden Hyperparameter_Tuning.zip sample_data

	zip warning: name not matched: Hyperparameter_Tuning.zip
  adding: all_in_one/ (stored 0%)
  adding: all_in_one/untitled_project/ (stored 0%)
  adding: all_in_one/untitled_project/trial_3/ (stored 0%)
  adding: all_in_one/untitled_project/trial_3/checkpoint.weights.h5 (deflated 26%)
  adding: all_in_one/untitled_project/trial_3/build_config.json (stored 0%)
  adding: all_in_one/untitled_project/trial_3/trial.json (deflated 64%)
  adding: all_in_one/untitled_project/tuner0.json (stored 0%)
  adding: all_in_one/untitled_project/trial_2/ (stored 0%)
  adding: all_in_one/untitled_project/trial_2/checkpoint.weights.h5 (deflated 88%)
  adding: all_in_one/untitled_project/trial_2/build_config.json (stored 0%)
  adding: all_in_one/untitled_project/trial_2/trial.json (deflated 65%)
  adding: all_in_one/untitled_project/trial_4/ (stored 0%)
  adding: all_in_one/untitled_project/trial_4/checkpoint.weights.h5 (deflated 23%)
  adding: all_in_one/untitled_project/trial_4/build_config.json (stored 0

In [81]:
!ls -lh ANN_Project.zip

-rw-r--r-- 1 root root 20M Aug 20 14:38 ANN_Project.zip


In [82]:
from google.colab import files
files.download('ANN_Project.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>